## 1. setup
imports and the random seed. run the cells from top to bottom, because later cells use variables made by the earlier ones.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
import os



SEED = 42

## 2. load sentiment data
sentiment data, split 80/20 with the same proportion of each label.

In [2]:
sent_df = pd.read_csv("comp-472-project-fall-2026-concordia/sentiment_train.csv")

X_sent = sent_df.drop(columns=["id", "label"]).to_numpy()
y_sent = sent_df["label"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X_sent, y_sent,
    test_size=0.2,
    stratify=y_sent,
    random_state=SEED,
)

## 3. load image data
image data, split the same way as sentiment.

In [3]:
#image data, same thing as sentiment
img_df = pd.read_csv("comp-472-project-fall-2026-concordia/image_train.csv")

X_img = img_df.drop(columns=["id", "label"]).to_numpy()
y_img = img_df["label"].to_numpy()

X_img_train, X_img_val, y_img_train, y_img_val = train_test_split(
    X_img, y_img,
    test_size=0.2,
    stratify=y_img,
    random_state=SEED,
)

## 4. distance function
how far a row `x` is from every row in `X`.

In [4]:
def euclidean_distance(x, X):
    return np.sqrt(((X - x) ** 2).sum(axis=1))

## 5. k-means model
hand-written k-means, it never looks at the labels.

In [5]:
class KMeans:
    def __init__(self, k = 3, max_iter = 100, seed = SEED):
        self.k = k
        self.max_iter = max_iter
        self.seed = seed

    def _closest(self, X):
        #distance from every row to each centroid, shape (n_rows, k)
        dists = []
        for c in self.centroids:
            dists.append(euclidean_distance(c, X))
        dists = np.stack(dists, axis=1)
        return dists.argmin(axis=1), dists

    def fit(self, X):
        #start: pick k random rows as the first centroids
        rng = np.random.default_rng(self.seed)
        start = rng.choice(len(X), self.k, replace=False)
        self.centroids = X[start].copy()

        for i in range(self.max_iter):
            #step 1: assign each row to the closest centroid
            labels, dists = self._closest(X)

            #step 2: move each centroid to the mean of its rows
            new_centroids = self.centroids.copy()
            for j in range(self.k):
                rows = X[labels == j]
                if len(rows) > 0:
                    new_centroids[j] = rows.mean(axis=0)

            #step 3: stop early if the centroids did not move
            moved = not np.allclose(new_centroids, self.centroids)
            self.centroids = new_centroids
            if not moved:
                break

        #final assignment with the final centroids
        self.labels, dists = self._closest(X)
        self.n_iter = i + 1
        return self

    def predict(self, X):
        #nearest centroid for each row
        labels, dists = self._closest(X)
        return labels

    def inertia(self, X):
        #sum of squared distances of each row to its own centroid
        labels, dists = self._closest(X)
        total = 0
        for j in range(self.k):
            total += (dists[labels == j, j] ** 2).sum()
        return total

## 6. turning clusters into labels
each cluster gets the most common training label inside it, then we can measure accuracy.

In [6]:
#kmeans does not use labels, so we only use them here to name each cluster
def cluster_to_label(cluster_ids, y):
    mapping = {}
    for c in np.unique(cluster_ids):
        mapping[c] = Counter(y[cluster_ids == c]).most_common(1)[0][0]
    return mapping

def cluster_accuracy(clusters, y, mapping):
    #clusters with no training rows get label 0
    preds = np.array([mapping.get(c, 0) for c in clusters])
    return np.mean(preds == y)

## 7. first try: sentiment, k = 3
a single run with 3 clusters, one for each sentiment label.

In [7]:
#sentiment: 3 clusters
km = KMeans(k=3).fit(X_train)
mapping = cluster_to_label(km.labels, y_train)

train_acc = cluster_accuracy(km.labels, y_train, mapping)
val_acc = cluster_accuracy(km.predict(X_val), y_val, mapping)

print(f"iterations: {km.n_iter}")
print(f"inertia: {km.inertia(X_train):.2f}")
print(f"train accuracy: {train_acc:.1%}")
print(f"val accuracy: {val_acc:.1%}")
print("rows per cluster:", np.bincount(km.labels, minlength=km.k))
print("cluster -> label:", mapping)

iterations: 24
inertia: 252264.60
train accuracy: 39.6%
val accuracy: 41.0%
rows per cluster: [3689 2873 3038]
cluster -> label: {np.int64(0): np.int64(2), np.int64(1): np.int64(1), np.int64(2): np.int64(0)}


## 8. first try: image, k = 50
a single run with 50 clusters.

In [8]:
#image: 50 clusters
km = KMeans(k=50).fit(X_img_train)
mapping = cluster_to_label(km.labels, y_img_train)

train_acc = cluster_accuracy(km.labels, y_img_train, mapping)
val_acc = cluster_accuracy(km.predict(X_img_val), y_img_val, mapping)

print(f"iterations: {km.n_iter}")
print(f"inertia: {km.inertia(X_img_train):.2f}")
print(f"train accuracy: {train_acc:.1%}")
print(f"val accuracy: {val_acc:.1%}")
print("rows per cluster:", np.bincount(km.labels, minlength=km.k))

iterations: 38
inertia: 11684.46
train accuracy: 11.0%
val accuracy: 9.8%
rows per cluster: [304 134 184 447 245 302 443 183 289 373 241 379 289 324 275 411 245  97
 133 469 327 102 172 430 560 131 155 236 320 349 288 365 381 801 377 397
 273 392 402 388 420 401 468 207 392 253 235 276 373 362]


## 9. old k sweep (no scaling)
tries a few values of k on the raw features, this cell also defines `output_folder` and `unique_path`.

In [9]:
#try different k for each dataset, more clusters than classes is fine
#because many clusters can get the same label
output_folder = "/Users/roro/Desktop/kaggel_project/output"
os.makedirs(output_folder, exist_ok=True)

max_iter = 30
sent_k_values = [3, 5, 10, 20]
img_k_values = [50, 75, 100]


def unique_path(folder, base_name, ext=".csv"):
    path = os.path.join(folder, base_name + ext)
    n = 1
    while os.path.exists(path):
        path = os.path.join(folder, f"{base_name}_{n}{ext}")
        n += 1
    return path


results = []
datasets = [("sentiment", X_train, y_train, X_val, y_val, sent_k_values),
            ("image", X_img_train, y_img_train, X_img_val, y_img_val, img_k_values)]

for name, Xtr, ytr, Xva, yva, k_values in datasets:
    for k in k_values:
        km = KMeans(k=k, max_iter=max_iter).fit(Xtr)
        mapping = cluster_to_label(km.labels, ytr)
        train_acc = cluster_accuracy(km.labels, ytr, mapping)
        val_acc = cluster_accuracy(km.predict(Xva), yva, mapping)
        results.append({"dataset": name, "k": k, "inertia": km.inertia(Xtr),
                        "train_accuracy": train_acc, "val_accuracy": val_acc})
        print(f"{name}  k={k}  inertia={km.inertia(Xtr):.2f}  train acc={train_acc:.1%}  val acc={val_acc:.1%}")

sent_best = max([r for r in results if r["dataset"] == "sentiment"], key=lambda r: r["val_accuracy"])
img_best = max([r for r in results if r["dataset"] == "image"], key=lambda r: r["val_accuracy"])

save_path = unique_path(output_folder, "kmeans_k_sweep")
pd.DataFrame(results).to_csv(save_path, index=False)

print("saved to:", save_path)
print(f"best sentiment: k = {sent_best['k']} ({sent_best['val_accuracy']:.1%})")
print(f"best image: k = {img_best['k']} ({img_best['val_accuracy']:.1%})")

sentiment  k=3  inertia=252264.60  train acc=39.6%  val acc=41.0%
sentiment  k=5  inertia=243191.59  train acc=42.1%  val acc=41.2%
sentiment  k=10  inertia=231201.96  train acc=45.0%  val acc=45.2%
sentiment  k=20  inertia=220969.04  train acc=46.3%  val acc=47.2%
image  k=50  inertia=11684.79  train acc=11.0%  val acc=9.6%
image  k=75  inertia=11492.53  train acc=12.1%  val acc=10.2%
image  k=100  inertia=11365.80  train acc=13.1%  val acc=10.4%
saved to: /Users/roro/Desktop/kaggel_project/output/kmeans_k_sweep_1.csv
best sentiment: k = 20 (47.2%)
best image: k = 100 (10.4%)


## 10. feature scaling
the same scaling ideas that helped KNN, each one is learned on the training rows only and then applied to the other rows.

In [10]:
from sklearn.decomposition import PCA

def no_scaling(X_train, X_other):
    return X_train, X_other

def standardize(X_train, X_other):
    #mean and std come from the training rows only
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1
    return (X_train - mean) / std, (X_other - mean) / std

def sqrt_scaling(X_train, X_other):
    #for the image histograms: shrinks big counts so rare visual words matter too
    A = np.sqrt(X_train)
    B = np.sqrt(X_other)
    len_A = np.linalg.norm(A, axis=1, keepdims=True)
    len_B = np.linalg.norm(B, axis=1, keepdims=True)
    len_A[len_A == 0] = 1
    len_B[len_B == 0] = 1
    return A / len_A, B / len_B

def pca_whiten(X_train, X_other, n_components=64):
    #squeezes 768 features to 64 and gives each the same spread
    pca = PCA(n_components, whiten=True, random_state=SEED)
    pca.fit(X_train)
    return pca.transform(X_train), pca.transform(X_other)

## 11. several random starts
k-means depends on which rows it starts from, so we run it a few times with different seeds and keep the run with the tightest groups.

In [11]:
def fit_best_of(X, k, n_starts, max_iter=30):
    #run k-means n_starts times, each with a different starting seed,
    #and keep the run with the lowest inertia (tightest groups)
    #note: we pick by inertia, not accuracy, because k-means is not allowed to look at labels
    best_model = None
    best_inertia = None
    for s in range(n_starts):
        km = KMeans(k=k, max_iter=max_iter, seed=s).fit(X)
        inertia = km.inertia(X)
        if best_inertia is None or inertia < best_inertia:
            best_inertia = inertia
            best_model = km
    return best_model

## 12. quick test: a small slice of the real data
before spending minutes on all 9,600 / 16,000 rows, we try 1,000 training rows so each run takes seconds and we can see which direction things go. validation stays the full validation set.

In [12]:
#only the training rows get smaller, the validation rows stay the same
import time

n_small = 1000
n_starts = 1
X_small, y_small = X_train[:n_small], y_train[:n_small]
X_img_small, y_img_small = X_img_train[:n_small], y_img_train[:n_small]

#name, train rows, train labels, val rows, val labels, scalings, k values
small_datasets = [("sentiment", X_small, y_small, X_val, y_val, [("none", no_scaling), ("pca_whiten", pca_whiten)], [3, 20]),
                  ("image", X_img_small, y_img_small, X_img_val, y_img_val, [("none", no_scaling), ("sqrt", sqrt_scaling)], [50, 100])]

for name, Xtr, ytr, Xva, yva, scalings, k_values in small_datasets:
    for scaling_name, scaling_fn in scalings:
        X_a, X_b = scaling_fn(Xtr, Xva)
        for k in k_values:
            start_time = time.time()
            km = fit_best_of(X_a, k, n_starts)
            mapping = cluster_to_label(km.labels, ytr)
            val_acc = cluster_accuracy(km.predict(X_b), yva, mapping)
            seconds = time.time() - start_time
            print(f"{name}  {scaling_name}  k={k}  val acc={val_acc:.1%}  {seconds:.1f} seconds")

sentiment  none  k=3  val acc=38.4%  0.0 seconds
sentiment  none  k=20  val acc=45.5%  0.2 seconds
sentiment  pca_whiten  k=3  val acc=39.8%  0.0 seconds
sentiment  pca_whiten  k=20  val acc=44.1%  0.0 seconds
image  none  k=50  val acc=4.3%  0.3 seconds
image  none  k=100  val acc=6.5%  0.5 seconds
image  sqrt  k=50  val acc=4.3%  0.2 seconds
image  sqrt  k=100  val acc=6.3%  0.5 seconds


what to look for
- does scaling help?
- does a bigger k help?
- each run's seconds tell you roughly how long the full run will take (the full data is about 10 to 16 times more rows)

## 13. experiment A: scaling
same k for every scaling (the best k of the old sweep: 20 for sentiment, 100 for image), one start each, so only the scaling changes.

In [13]:
#estimated runtime: about 2 to 3 minutes
#(old notebook with the whole k sweep took about 1m40s, image k=100 with 30 iterations is the slowest fit,
#and here there are 3 fits of it plus 3 fast sentiment fits, standardize and pca add a few seconds)
n_starts = 1

sent_scalings = [("none", no_scaling), ("standardize", standardize), ("pca_whiten", pca_whiten)]
img_scalings = [("none", no_scaling), ("standardize", standardize), ("sqrt", sqrt_scaling)]

#name, train rows, train labels, val rows, val labels, scalings, k
scaling_datasets = [("sentiment", X_train, y_train, X_val, y_val, sent_scalings, 20),
                    ("image", X_img_train, y_img_train, X_img_val, y_img_val, img_scalings, 100)]

scaling_results = []
for name, Xtr, ytr, Xva, yva, scalings, k in scaling_datasets:
    for scaling_name, scaling_fn in scalings:
        #the scaling is learned on the train part only
        X_a, X_b = scaling_fn(Xtr, Xva)
        km = fit_best_of(X_a, k, n_starts)
        mapping = cluster_to_label(km.labels, ytr)
        train_acc = cluster_accuracy(km.labels, ytr, mapping)
        val_acc = cluster_accuracy(km.predict(X_b), yva, mapping)
        scaling_results.append({"dataset": name, "scaling": scaling_name, "k": k,
                                "inertia": km.inertia(X_a), "train_accuracy": train_acc, "val_accuracy": val_acc})
        print(f"{name}  {scaling_name}  k={k}  train acc={train_acc:.1%}  val acc={val_acc:.1%}")

#best scaling for each dataset (by validation accuracy)
sent_best_scaling = max([r for r in scaling_results if r["dataset"] == "sentiment"], key=lambda r: r["val_accuracy"])["scaling"]
img_best_scaling = max([r for r in scaling_results if r["dataset"] == "image"], key=lambda r: r["val_accuracy"])["scaling"]

save_path = unique_path(output_folder, "kmeans_experiment_scaling")
pd.DataFrame(scaling_results).to_csv(save_path, index=False)

print("saved to:", save_path)
print("best sentiment scaling:", sent_best_scaling)
print("best image scaling:", img_best_scaling)

sentiment  none  k=20  train acc=48.0%  val acc=45.8%
sentiment  standardize  k=20  train acc=47.8%  val acc=47.5%
sentiment  pca_whiten  k=20  train acc=45.9%  val acc=45.1%
image  none  k=100  train acc=13.3%  val acc=10.8%
image  standardize  k=100  train acc=13.2%  val acc=10.5%
image  sqrt  k=100  train acc=14.2%  val acc=12.2%
saved to: /Users/roro/Desktop/kaggel_project/output/kmeans_experiment_scaling.csv
best sentiment scaling: standardize
best image scaling: sqrt


## 14. experiment B: number of clusters k (with the best scaling)
more clusters than labels is allowed, each cluster gets the most common label in it. but too many clusters starts to act like memorizing.

In [14]:
#estimated runtime: about 3 to 5 minutes
#(time grows with k: image k=50, 100, 200 is roughly 15s + 30s + 60s,
#sentiment k up to 80 is faster, about a minute for all 5 values)
n_starts = 1
scaling_by_name = {"none": no_scaling, "standardize": standardize, "sqrt": sqrt_scaling, "pca_whiten": pca_whiten}

sent_k_values = [3, 10, 20, 40, 80]
img_k_values = [50, 100, 200]

k_datasets = [("sentiment", X_train, y_train, X_val, y_val, sent_k_values, sent_best_scaling),
              ("image", X_img_train, y_img_train, X_img_val, y_img_val, img_k_values, img_best_scaling)]

k_results = []
for name, Xtr, ytr, Xva, yva, k_values, scaling_name in k_datasets:
    #the scaling is learned on the train part only, so it is the same for every k
    X_a, X_b = scaling_by_name[scaling_name](Xtr, Xva)
    for k in k_values:
        km = fit_best_of(X_a, k, n_starts)
        mapping = cluster_to_label(km.labels, ytr)
        train_acc = cluster_accuracy(km.labels, ytr, mapping)
        val_acc = cluster_accuracy(km.predict(X_b), yva, mapping)
        k_results.append({"dataset": name, "scaling": scaling_name, "k": k,
                          "inertia": km.inertia(X_a), "train_accuracy": train_acc, "val_accuracy": val_acc})
        print(f"{name}  {scaling_name}  k={k}  train acc={train_acc:.1%}  val acc={val_acc:.1%}")

sent_best_k = max([r for r in k_results if r["dataset"] == "sentiment"], key=lambda r: r["val_accuracy"])["k"]
img_best_k = max([r for r in k_results if r["dataset"] == "image"], key=lambda r: r["val_accuracy"])["k"]

save_path = unique_path(output_folder, "kmeans_experiment_k")
pd.DataFrame(k_results).to_csv(save_path, index=False)

print("saved to:", save_path)
print("best sentiment k:", sent_best_k)
print("best image k:", img_best_k)

sentiment  standardize  k=3  train acc=39.2%  val acc=38.5%
sentiment  standardize  k=10  train acc=44.7%  val acc=45.0%
sentiment  standardize  k=20  train acc=47.8%  val acc=47.5%
sentiment  standardize  k=40  train acc=49.5%  val acc=48.2%
sentiment  standardize  k=80  train acc=49.9%  val acc=46.8%
image  sqrt  k=50  train acc=10.0%  val acc=8.9%
image  sqrt  k=100  train acc=14.2%  val acc=12.2%
image  sqrt  k=200  train acc=16.9%  val acc=13.9%
saved to: /Users/roro/Desktop/kaggel_project/output/kmeans_experiment_k.csv
best sentiment k: 40
best image k: 200


## 15. experiment C: several random starts
compares 1 start with 5 starts, using the best scaling and the best k. the start is chosen by inertia, the labels are only used to measure accuracy.

In [15]:
#estimated runtime: about 4 to 8 minutes
#(5 starts take 5 times as long as 1 fit, so each dataset is 6 fits in total;
#the image data is the slow part and depends on the best k from experiment B)
starts_values = [1, 5]

starts_datasets = [("sentiment", X_train, y_train, X_val, y_val, sent_best_scaling, sent_best_k),
                   ("image", X_img_train, y_img_train, X_img_val, y_img_val, img_best_scaling, img_best_k)]

starts_results = []
for name, Xtr, ytr, Xva, yva, scaling_name, k in starts_datasets:
    X_a, X_b = scaling_by_name[scaling_name](Xtr, Xva)
    for n_starts in starts_values:
        km = fit_best_of(X_a, k, n_starts)
        mapping = cluster_to_label(km.labels, ytr)
        val_acc = cluster_accuracy(km.predict(X_b), yva, mapping)
        starts_results.append({"dataset": name, "scaling": scaling_name, "k": k, "n_starts": n_starts,
                               "inertia": km.inertia(X_a), "val_accuracy": val_acc})
        print(f"{name}  {scaling_name}  k={k}  starts={n_starts}  inertia={km.inertia(X_a):.2f}  val acc={val_acc:.1%}")

#more starts only wins if it is strictly better, if equal we keep 1 because it is faster
best_starts = {}
best_val_acc = {}
for name in ["sentiment", "image"]:
    best_starts[name] = 1
    best_val_acc[name] = 0
    for r in starts_results:
        if r["dataset"] == name:
            if best_val_acc[name] == 0 or r["val_accuracy"] > best_val_acc[name]:
                best_starts[name] = r["n_starts"]
                best_val_acc[name] = r["val_accuracy"]
sent_best_starts = best_starts["sentiment"]
img_best_starts = best_starts["image"]

save_path = unique_path(output_folder, "kmeans_experiment_starts")
pd.DataFrame(starts_results).to_csv(save_path, index=False)

print("saved to:", save_path)
print("best sentiment starts:", sent_best_starts)
print("best image starts:", img_best_starts)

sentiment  standardize  k=40  starts=1  inertia=5828447.64  val acc=48.2%
sentiment  standardize  k=40  starts=5  inertia=5804661.61  val acc=49.1%
image  sqrt  k=200  starts=1  inertia=11464.13  val acc=13.9%
image  sqrt  k=200  starts=5  inertia=11457.74  val acc=14.0%
saved to: /Users/roro/Desktop/kaggel_project/output/kmeans_experiment_starts.csv
best sentiment starts: 5
best image starts: 5


## 16. final model and kaggle submission (with the best settings)
refit on all training rows with the best scaling, k and number of starts, predict the test set and write the file.

In [16]:
#estimated runtime: about 1 to 8 minutes, one fit on all the rows (a bit slower than on the train part)
#if the best number of starts is 5, it is 5 fits per dataset
sent_test = pd.read_csv("comp-472-project-fall-2026-concordia/sentiment_test.csv")
img_test = pd.read_csv("comp-472-project-fall-2026-concordia/image_test.csv")

#the scaling is learned on all the training rows, then applied to the test rows
X_a, X_b = scaling_by_name[sent_best_scaling](X_sent, sent_test.drop(columns=["id"]).to_numpy())
sent_model = fit_best_of(X_a, sent_best_k, sent_best_starts)
sent_mapping = cluster_to_label(sent_model.labels, y_sent)
sent_clusters = sent_model.predict(X_b)
sentiment_predictions = [sent_mapping.get(c, 0) for c in sent_clusters]

X_a, X_b = scaling_by_name[img_best_scaling](X_img, img_test.drop(columns=["id"]).to_numpy())
img_model = fit_best_of(X_a, img_best_k, img_best_starts)
img_mapping = cluster_to_label(img_model.labels, y_img)
img_clusters = img_model.predict(X_b)
image_predictions = [img_mapping.get(c, 0) for c in img_clusters]

#same as the code on the kaggle page
sample = pd.read_csv("comp-472-project-fall-2026-concordia/sample_submission.csv")

image_results = pd.DataFrame({"id": img_test["id"], "label": image_predictions})
sentiment_results = pd.DataFrame({"id": sent_test["id"], "label": sentiment_predictions})
predictions = pd.concat([image_results, sentiment_results], ignore_index=True)

submission = sample[["id"]].merge(predictions, on="id", how="left", validate="one_to_one")
assert len(submission) == len(sample)
assert submission["label"].notna().all()
submission["label"] = submission["label"].astype(int)

save_path = unique_path(output_folder, "submission_kmeans")
submission.to_csv(save_path, index=False)

print("saved to:", save_path)
print(f"sentiment: {sent_best_scaling} k = {sent_best_k} starts = {sent_best_starts} ({best_val_acc['sentiment']:.1%} on validation)")
print(f"image: {img_best_scaling} k = {img_best_k} starts = {img_best_starts} ({best_val_acc['image']:.1%} on validation)")

saved to: /Users/roro/Desktop/kaggel_project/output/submission_kmeans_1.csv
sentiment: standardize k = 40 starts = 5 (49.1% on validation)
image: sqrt k = 200 starts = 5 (14.0% on validation)
